In [1]:
import pandas as pd
import os
from datetime import datetime
import numpy as np

def process_air_quality_data(
    input_file_path,
    output_dir="processed_data",
    date_format="%Y%m%d"
):
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"正在处理文件: {input_file_path}")
    
    try:
        # 读取CSV文件
        df = pd.read_csv(input_file_path)
        print(f"原始数据形状: {df.shape}")
        
        # 查看前几列的名称以确认列结构
        print(f"列名: {df.columns.tolist()[:10]}...")
        
        # 确保基本列存在
        required_cols = ['date', 'hour', 'type']
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"未找到必需的列: {col}")
        
        # 识别所有站点列
        station_cols = [col for col in df.columns if col not in required_cols]
        n_stations = len(station_cols)
        print(f"发现{n_stations}个监测站")
        
        # 检查数据中的日期范围
        unique_dates = df['date'].unique()
        print(f"数据包含{len(unique_dates)}个日期")
        
        # 对每个日期分别处理
        for date_val in unique_dates:
            # 转换日期格式
            try:
                # 将整数日期转换为字符串
                if isinstance(date_val, (int, float)):
                    date_val = str(int(date_val))
                
                # 解析日期
                date_obj = datetime.strptime(date_val, date_format)
                formatted_date = date_obj.strftime("%Y-%m-%d")
            except:
                print(f"警告: 无法解析日期 {date_val}，使用原始值")
                formatted_date = str(date_val)
            
            print(f"处理日期: {formatted_date}")
            
            # 筛选当前日期的数据
            date_df = df[df['date'] == date_val]
            
            # 创建DataFrame来保存结果
            particulate_df = pd.DataFrame()
            gas_df = pd.DataFrame()
            
            # 处理不同的污染物类型
            pm25_rows = date_df[date_df['type'] == 'PM2.5']
            pm10_rows = date_df[date_df['type'] == 'PM10']
            aqi_rows = date_df[date_df['type'] == 'AQI']
            
            # 查找气体类型
            gas_types = [t for t in date_df['type'].unique() 
                        if t not in ['PM2.5', 'PM10', 'AQI', 'PM2.5_24h', 'PM10_24h']]
            
            print(f"发现的污染物类型: {date_df['type'].unique().tolist()}")
            
            # 1. 准备颗粒物数据集
            # 首先创建基础DataFrame，包含所有站点和时间戳组合
            all_hours = date_df['hour'].unique()
            rows = []
            
            for station in station_cols:
                for hour in all_hours:
                    row = {
                        'station_id': station,
                        'timestamp': hour
                    }
                    
                    # 添加PM2.5值
                    pm25_hour_data = pm25_rows[pm25_rows['hour'] == hour]
                    if not pm25_hour_data.empty and station in pm25_hour_data.columns:
                        row['PM2.5'] = pm25_hour_data[station].values[0]
                    
                    # 添加PM10值
                    pm10_hour_data = pm10_rows[pm10_rows['hour'] == hour]
                    if not pm10_hour_data.empty and station in pm10_hour_data.columns:
                        row['PM10'] = pm10_hour_data[station].values[0]
                    
                    # 添加AQI值
                    aqi_hour_data = aqi_rows[aqi_rows['hour'] == hour]
                    if not aqi_hour_data.empty and station in aqi_hour_data.columns:
                        row['AQI'] = aqi_hour_data[station].values[0]
                    
                    rows.append(row)
            
            # 创建颗粒物DataFrame
            particulate_df = pd.DataFrame(rows)
            
            # 2. 准备气体数据集
            gas_rows = []
            
            for station in station_cols:
                for hour in all_hours:
                    row = {
                        'station_id': station,
                        'timestamp': hour
                    }
                    
                    # 添加各类气体污染物的值
                    for gas_type in gas_types:
                        gas_data = date_df[(date_df['type'] == gas_type) & (date_df['hour'] == hour)]
                        if not gas_data.empty and station in gas_data.columns:
                            row[gas_type] = gas_data[station].values[0]
                    
                    gas_rows.append(row)
            
            # 创建气体DataFrame
            gas_df = pd.DataFrame(gas_rows)
            
            # 添加日期列
            particulate_df['date'] = formatted_date
            gas_df['date'] = formatted_date
            
            # 对结果进行排序
            particulate_df = particulate_df.sort_values(['station_id', 'timestamp']).reset_index(drop=True)
            gas_df = gas_df.sort_values(['station_id', 'timestamp']).reset_index(drop=True)
            
            print(f"颗粒物数据形状: {particulate_df.shape}")
            print(f"气体数据形状: {gas_df.shape}")
            
            # 保存处理后的结果
            particulate_file = os.path.join(output_dir, f"{formatted_date}_particulate.csv")
            gas_file = os.path.join(output_dir, f"{formatted_date}_gas.csv")
            
            particulate_df.to_csv(particulate_file, index=False)
            gas_df.to_csv(gas_file, index=False)
            
            print(f"已保存到: {particulate_file} 和 {gas_file}")
        
        return True
    
    except Exception as e:
        print(f"处理文件时出错: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

def process_monthly_files(
    base_dir="E:/24-25-2/环境与资源保护法/大作业数据分析/beijing_20250101-20250510",
    pattern="beijing_all_",
    year=2025,
    month=1,
    output_dir="processed_data"
):
    
    print(f"处理{year}年{month}月的所有数据文件...")
    
    # 构建月份字符串 (例如: '01' 对应 1月)
    month_str = f"{month:02d}"
    
    # 列出目录中的所有文件
    all_files = os.listdir(base_dir)
    
    # 筛选出匹配的文件
    target_pattern = f"{pattern}{year}{month_str}"
    month_files = [f for f in all_files if target_pattern in f and f.endswith('.csv')]
    
    if not month_files:
        print(f"未找到{year}年{month}月的数据文件!")
        return []
    
    print(f"找到{len(month_files)}个文件: {', '.join(month_files[:5])}...")
    
    # 处理每个文件
    results = []
    for file in month_files:
        file_path = os.path.join(base_dir, file)
        success = process_air_quality_data(file_path, output_dir)
        results.append({
            'file': file,
            'success': success
        })
    
    # 输出处理结果摘要
    success_count = sum(1 for r in results if r['success'])
    print(f"\n处理完成! 成功: {success_count}/{len(results)} 个文件")
    
    # 显示失败的文件
    failed_files = [r['file'] for r in results if not r['success']]
    if failed_files:
        print(f"失败的文件: {', '.join(failed_files)}")
    
    return results

def validate_processed_files(output_dir="processed_data"):
   
    
    print(f"\n验证处理后的文件...")
    
    if not os.path.exists(output_dir):
        print(f"错误: 目录{output_dir}不存在!")
        return
    
    all_files = os.listdir(output_dir)
    particulate_files = [f for f in all_files if 'particulate' in f]
    gas_files = [f for f in all_files if 'gas' in f]
    
    print(f"找到{len(particulate_files)}个颗粒物数据文件和{len(gas_files)}个气体数据文件")
    
    # 检查文件对应关系
    all_dates = set()
    for f in particulate_files:
        date_part = f.split('_particulate')[0]
        all_dates.add(date_part)
    
    gas_dates = set()
    for f in gas_files:
        date_part = f.split('_gas')[0]
        gas_dates.add(date_part)
    
    missing_gas = all_dates - gas_dates
    missing_particulate = gas_dates - all_dates
    
    if missing_gas:
        print(f"警告: 以下日期缺少气体数据文件: {missing_gas}")
    
    if missing_particulate:
        print(f"警告: 以下日期缺少颗粒物数据文件: {missing_particulate}")
    
    # 检查随机选择的文件内容
    import random
    
    if particulate_files:
        sample_file = random.choice(particulate_files)
        print(f"\n检查颗粒物样本文件: {sample_file}")
        try:
            df = pd.read_csv(os.path.join(output_dir, sample_file))
            print(f"  形状: {df.shape}")
            print(f"  列: {df.columns.tolist()}")
            print(f"  站点数: {df['station_id'].nunique()}")
            print(f"  时间步数: {df['timestamp'].nunique()}")
            # 检查数据完整性
            station_coverage = df.groupby('station_id')['timestamp'].nunique() / 24 * 100
            print(f"  站点时间覆盖率: {station_coverage.mean():.2f}% (平均)")
            if 'PM2.5' in df.columns:
                print(f"  PM2.5非空值比例: {df['PM2.5'].notna().mean()*100:.2f}%")
            if 'PM10' in df.columns:
                print(f"  PM10非空值比例: {df['PM10'].notna().mean()*100:.2f}%")
            if 'AQI' in df.columns:
                print(f"  AQI非空值比例: {df['AQI'].notna().mean()*100:.2f}%")
        except Exception as e:
            print(f"  检查时出错: {str(e)}")
    
    if gas_files:
        sample_file = random.choice(gas_files)
        print(f"\n检查气体样本文件: {sample_file}")
        try:
            df = pd.read_csv(os.path.join(output_dir, sample_file))
            print(f"  形状: {df.shape}")
            print(f"  列: {df.columns.tolist()}")
            print(f"  站点数: {df['station_id'].nunique()}")
            print(f"  时间步数: {df['timestamp'].nunique()}")
            # 检查数据完整性
            station_coverage = df.groupby('station_id')['timestamp'].nunique() / 24 * 100
            print(f"  站点时间覆盖率: {station_coverage.mean():.2f}% (平均)")
            # 检查常见气体污染物
            for col in df.columns:
                if col not in ['station_id', 'timestamp', 'date']:
                    print(f"  {col}非空值比例: {df[col].notna().mean()*100:.2f}%")
        except Exception as e:
            print(f"  检查时出错: {str(e)}")

# 单文件处理函数
def process_single_file(
    file_path,
    output_dir="processed_data"
):

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    success = process_air_quality_data(file_path, output_dir)
    
    if success:
        print(f"文件 {os.path.basename(file_path)} 处理成功!")
    else:
        print(f"文件 {os.path.basename(file_path)} 处理失败!")
    
    # 验证处理结果
    validate_processed_files(output_dir)
    
    return success

# 执行
if __name__ == "__main__":
    # 设置参数
    BASE_DIR = "E:/24-25-2/环境与资源保护法/大作业数据分析/beijing_20250101-20250510"
    OUTPUT_DIR = "data/air_quality" 
    
    # 创建输出目录
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 处理方式选择
    import sys
    
    # 默认处理整个月
    process_all = True
    
    if len(sys.argv) > 1:
        specific_file = sys.argv[1]
        if os.path.exists(specific_file):
            process_all = False
            process_single_file(specific_file, OUTPUT_DIR)
    
    if process_all:
        # 处理1月份的数据文件
        process_monthly_files(
            base_dir=BASE_DIR,
            pattern="beijing_all_",
            year=2025,
            month=1,
            output_dir=OUTPUT_DIR
        )
        
        # 验证处理后的文件
        validate_processed_files(OUTPUT_DIR)
    
    print("\n数据预处理完成! 可以开始使用PyPOTS进行时间序列插补")

处理2025年1月的所有数据文件...
找到31个文件: beijing_all_20250101.csv, beijing_all_20250102.csv, beijing_all_20250103.csv, beijing_all_20250104.csv, beijing_all_20250105.csv...
正在处理文件: E:/24-25-2/环境与资源保护法/大作业数据分析/beijing_20250101-20250510\beijing_all_20250101.csv
原始数据形状: (120, 38)
列名: ['date', 'hour', 'type', '东城东四', '东城天坛', '西城官园', '西城万寿西宫', '朝阳奥体中心', '朝阳农展馆', '海淀万柳']...
发现35个监测站
数据包含1个日期
警告: 无法解析日期 20250101，使用原始值
处理日期: 20250101
发现的污染物类型: ['PM2.5', 'PM2.5_24h', 'PM10', 'PM10_24h', 'AQI']
颗粒物数据形状: (840, 6)
气体数据形状: (840, 3)
已保存到: data/air_quality\20250101_particulate.csv 和 data/air_quality\20250101_gas.csv
正在处理文件: E:/24-25-2/环境与资源保护法/大作业数据分析/beijing_20250101-20250510\beijing_all_20250102.csv
原始数据形状: (120, 38)
列名: ['date', 'hour', 'type', '东城东四', '东城天坛', '西城官园', '西城万寿西宫', '朝阳奥体中心', '朝阳农展馆', '海淀万柳']...
发现35个监测站
数据包含1个日期
警告: 无法解析日期 20250102，使用原始值
处理日期: 20250102
发现的污染物类型: ['PM2.5', 'PM2.5_24h', 'PM10', 'PM10_24h', 'AQI']
颗粒物数据形状: (840, 6)
气体数据形状: (840, 3)
已保存到: data/air_quality\20250102_particulate.csv 和 d